In [95]:
from langgraph.graph import StateGraph , START , END
from langchain_openrouter import ChatOpenRouter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage , HumanMessage
from pydantic import BaseModel , Field
from typing import TypedDict , Annotated , Literal
import operator
from dotenv import load_dotenv
load_dotenv()

True

In [96]:
# state
class TweetState(TypedDict):
    topic:str
    tweet:str
    evaluation:["approved" , "needs_improvement"]
    feedback:str
    iteration:int
    max_iteration:int
    tweet_history:Annotated[
        list[str] , operator.add
    ]
    feedback_history:Annotated[
        list[str] , operator.add
    ]


In [97]:
class TweetEvaluationSchema(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

In [98]:
generator_llm = ChatOpenRouter(
    model = "google/gemma-4-31b-it:free"
)

evaluator_llm = ChatOpenRouter(
    model = "nvidia/nemotron-3-ultra-550b-a55b:free"
)
strcutured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluationSchema)

optimizer_llm = ChatOpenRouter(
    model = "nvidia/nemotron-3.5-lightning:free"
)

"""generator_llm = ChatGoogleGenerativeAI(
    model = "gemini-3.6-flash"
)

evaluator_llm = ChatGoogleGenerativeAI(
    model = "gemini-3.6-flash"
)
strcutured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluationSchema)

optimizer_llm = ChatGoogleGenerativeAI(
    model = "gemini-3.6-flash"
)
"""

'generator_llm = ChatGoogleGenerativeAI(\n    model = "gemini-3.6-flash"\n)\n\nevaluator_llm = ChatGoogleGenerativeAI(\n    model = "gemini-3.6-flash"\n)\nstrcutured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluationSchema)\n\noptimizer_llm = ChatGoogleGenerativeAI(\n    model = "gemini-3.6-flash"\n)\n'

In [99]:
def generate_tweet(state:TweetState):
    tweet_topic = state["topic"]
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(
            content=f"""
                Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

                Rules:
                - Do NOT use question-answer format.
                - Max 280 characters.
                - Use observational humor, irony, sarcasm, or cultural references.
                - Think in meme logic, punchlines, or relatable takes.
                - Use simple, day to day english
            """
        )
    ]
    response = generator_llm.invoke(messages)
    return {
        "tweet" : response.content,
        "tweet_history" : [response.content]
    }

def evaluate_tweet(state:TweetState):
    messages = [

        SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),

        HumanMessage(
            content=f"""
                Evaluate the following tweet:

                Tweet: "{state['tweet']}"

                Use the criteria below to evaluate the tweet:

                1. Originality - Is this fresh, or have you seen it a hundred times before?  
                2. Humor - Did it genuinely make you smile, laugh, or chuckle?  
                3. Punchiness - Is it short, sharp, and scroll-stopping?  
                4. Virality Potential - Would people retweet or share it?  
                5. Format - Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

                Auto-reject if:
                - It's written in question-answer format (e.g., "Why did..." or "What happens when...")
                - It exceeds 280 characters
                - It reads like a traditional setup-punchline joke
                - Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

                ### Respond ONLY in structured format:
                - evaluation: "approved" or "needs_improvement"  
                - feedback: One paragraph explaining the strengths and weaknesses 
            """
        )
    ]

    response = strcutured_evaluator_llm.invoke(messages)

    return {
        "evaluation" : response.evaluation,
        "feedback" : response.feedback,
        "feedback_history" : [response.feedback]
    }    

    

def optimize_tweet(state:TweetState):

    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(
            content=f"""
                Improve the tweet based on this feedback:
                "{state['feedback']}"

                Topic: "{state['topic']}"
                Original Tweet:
                {state['tweet']}

                Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
            """
        )
    ]

    response = optimizer_llm.invoke(messages)

    iteration = state["iteration"] + 1

    return {
        "tweet" : response.content,
        "iteration" : iteration
    }

def route_evaluation(state: TweetState):

    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'



In [100]:
graph = StateGraph(TweetState)

# Nodes
graph.add_node("generate_tweet" , generate_tweet)
graph.add_node("evaluate_tweet" , evaluate_tweet)
graph.add_node("optimize_tweet" , optimize_tweet)

# Edges
graph.add_edge(START , "generate_tweet")
graph.add_edge("generate_tweet" , "evaluate_tweet")
graph.add_conditional_edges(
    "evaluate_tweet" , route_evaluation , 
    {
        "approved" : END , 
        "needs_improvement" : "optimize_tweet"
    }
)
graph.add_edge("optimize_tweet" , "evaluate_tweet")

workflow = graph.compile()

In [ ]:
initial_state = {
    "topic" : "indian railway",
    "iteration" : 1,
    "max_iteration" : 5
}

final_state = workflow.invoke(initial_state)

In [ ]:
final_state